# Capstone: LLM + Formal Verifier

## Capstone: LLM Reasoning with SMT Verification Layer

This capstone builds a hybrid reasoning pipeline:
1. LLM receives a mathematical/logical question
2. LLM generates a proposed solution and formal constraints
3. Z3 verifies whether the solution satisfies the constraints
4. If verification fails, the system provides feedback to the LLM for correction
5. Final answer includes the verified solution and proof

This pattern — LLM generates, solver verifies, feedback loop corrects — is the core pattern of neurosymbolic AI for reliable reasoning.

In [ ]:
from typing import Dict, Optional
import json

try:
    from z3 import *
    HAS_Z3 = True
except ImportError:
    HAS_Z3 = False

class LLMPlusVerifier:
    def __init__(self):
        self.max_attempts = 3
    
    def llm_generate(self, problem: str, feedback: Optional[str] = None) -> Dict:
        # Simulate LLM generating a solution + constraints
        solutions = {
            'schedule_3_tasks': {'t1': 9, 't2': 11, 't3': 14},
            'schedule_3_tasks_fixed': {'t1': 9, 't2': 12, 't3': 14},
        }
        key = 'schedule_3_tasks_fixed' if feedback else 'schedule_3_tasks'
        return solutions.get(key, {'t1': 9, 't2': 11, 't3': 12})
    
    def verify_solution(self, solution: Dict, constraints) -> Tuple[bool, str]:
        if not HAS_Z3:
            return True, 'Z3 not available, assuming valid'
        s = Solver()
        for c in constraints:
            s.add(c)
        result = s.check()
        return result == sat, str(result)
    
    def solve_with_verification(self, problem: str):
        print(f'Problem: {problem}')
        feedback = None
        
        for attempt in range(self.max_attempts):
            solution = self.llm_generate(problem, feedback)
            print(f'  Attempt {attempt+1}: LLM proposed {solution}')
            
            if HAS_Z3:
                t1 = IntVal(solution.get('t1', 0))
                t2 = IntVal(solution.get('t2', 0))
                t3 = IntVal(solution.get('t3', 0))
                constraints = [
                    t1 >= 9, t1 <= 17,
                    t2 >= 9, t2 <= 17,
                    t3 >= 9, t3 <= 17,
                    Or(t2 - t1 >= 2, t1 - t2 >= 2),
                    Or(t3 - t2 >= 1, t2 - t3 >= 1),
                ]
                valid, msg = self.verify_solution(solution, constraints)
            else:
                t1, t2, t3 = solution.get('t1',9), solution.get('t2',11), solution.get('t3',14)
                valid = (9<=t1<=17 and 9<=t2<=17 and 9<=t3<=17 and
                         abs(t2-t1)>=2 and abs(t3-t2)>=1)
                msg = 'verified manually'
            
            if valid:
                print(f'  Verified! Solution: {solution}')
                return solution
            else:
                feedback = f'Constraints not satisfied: {msg}'
                print(f'  Verification failed: {feedback}')
        
        return None

solver = LLMPlusVerifier()
result = solver.solve_with_verification('Schedule 3 meetings (t1,t2,t3) in work hours with min 2h gap between t1-t2 and 1h between t2-t3')
print(f'\nFinal verified solution: {result}')
